#### Libraries, plot settings, utils

In [ ]:
# ===========================================
# Imports and Setup
# ===========================================
import numpy as np
import torch
import matplotlib.pyplot as plt
import gc
from scipy.integrate import odeint

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float32)
gc.collect()
torch.cuda.empty_cache()

# Plotting defaults
plt.style.use('tableau-colorblind10')
plt.rcParams.update({
    "figure.autolayout": True,
    "font.size": 10,
    "font.family": "serif",
    "font.serif": "cmr10",
    "mathtext.fontset": "cm",
    "axes.formatter.use_mathtext": True,
    "axes.titlesize": 10,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9
})

# ===========================================
# Activation Functions
# ===========================================
def phi(x): return torch.tanh(x)
def phi_prime(x): return 1 - torch.tanh(x).square()

def phi_np(x): return np.tanh(x)
def dphi_np(x): return 1 - np.tanh(x)**2

# ===========================================
# Network Functions
# ===========================================
def inter_matrix(K, N, g, gamma):
    J = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            z = np.random.normal(size=2)
            J[i,j] = g/np.sqrt(K) * z[0]
            J[j,i] = g/np.sqrt(K) * (gamma*z[0] + np.sqrt(1-gamma**2)*z[1])
    return J

def adjacency_matrix(N, K, degree_sequence):
    A = np.zeros((N, N))
    for i in range(N):
        for j in range(i+1, N):
            p_ij = degree_sequence[i]*degree_sequence[j]/(K*N)
            if np.random.rand() < p_ij:
                A[i,j] = A[j,i] = 1
    return A

# ===========================================
# Dynamics
# ===========================================
def dynamics(x, t, W): return -x + np.dot(W, phi_np(x))
def dynamics_self_coup(x, t, W, s=2.5): return -x + np.dot(W, phi_np(x)) + s*phi_np(x)

def simulate_costum_degree(params, hyperparams, ic=None):
    N, g, n, mu, gamma, k_list, Pk_list = params
    N_steps, dt = hyperparams
    t = np.linspace(0, N_steps*dt, N_steps)

    degree_sequence = np.random.choice(k_list, p=Pk_list, size=N)
    A = adjacency_matrix(N, np.mean(degree_sequence), degree_sequence)
    K_true = np.mean(np.sum(A, axis=0))
    J = inter_matrix(int(K_true), N, g, gamma)
    W = A * J

    if ic is None:
        sim = odeint(dynamics, np.random.random(N)*2-1, t, args=(W,))
    else:
        sim = odeint(dynamics, ic, t, args=(W,))
    return degree_sequence, sim[:, -1]

# ===========================================
# DMFT Utilities
# ===========================================
def sample_multivariate_gaussian_torch(mean, cov, n_samples):
    eps = 1e-5 * torch.eye(cov.size(0), device=mean.device)
    L = torch.linalg.cholesky(cov + eps)
    z = torch.randn(n_samples, cov.size(0), device=mean.device)
    return mean + z @ L.T

def HWHM(acf_values):
    half_max = acf_values[0]/2
    indices = np.where(acf_values < half_max)[0]
    if len(indices)==0: return None
    i = indices[0]
    x0, x1 = i-1, i
    y0, y1 = acf_values[x0], acf_values[x1]
    return x0 + (half_max - y0)*(x1-x0)/(y1-y0)

def compute_corr_k(phi_x, k_list, Pk_list, K):
    N_k, N, T = phi_x.shape
    corr_k = torch.zeros((N_k, T, T), device=phi_x.device)
    corr = torch.zeros((T, T), device=phi_x.device)
    for i in range(N_k):
        ck = phi_x[i].transpose(0,1) @ phi_x[i] / N
        corr_k[i] = ck
        corr += (k_list[i]/K)*Pk_list[i]*ck
    return corr_k, corr

def compute_magnetization(phi_x, k_list, Pk_list, K):
    weights = (k_list/K)*Pk_list
    return (weights[:,None]*phi_x.mean(dim=1)).sum(dim=0)

def integ_G_k(x_k, G_k_old, G_old, dt, g, gamma, k_ratio, sqrt_k):
    N, T = x_k.shape
    G_k_temp = torch.zeros((N,T,T), device=x_k.device)
    for s in range(T-1):
        for t in range(s, T-1):
            x_slice = x_k[:, s:t]
            G_slice = G_old[t, s:t]
            G_k_slice = G_k_old[s:t, s]
            delta = gamma * g**2 * k_ratio * dt**2 * (phi_prime(x_slice) * G_slice @ G_k_slice)
            if t==s: delta += g*sqrt_k*dt
            G_k_temp[:, t+1, s] = G_k_old[t,s]*(1-dt) + delta
    return G_k_temp.mean(dim=0)

def compute_G_k(x, G_k_old, G_old, k_list, Pk_list, K, dt, g, gamma):
    N_k = x.shape[0]
    T = x.shape[2]
    G_k = torch.zeros((N_k,T,T), device=x.device)
    G_total = torch.zeros((T,T), device=x.device)
    sqrt_kK = torch.sqrt(k_list/K).unsqueeze(1)
    k_over_K = (k_list/K).unsqueeze(1)
    for i in range(N_k):
        G_k[i] = integ_G_k(x[i], G_k_old[i], G_old, dt, g, gamma, k_over_K[i].item(), sqrt_kK[i].item())
        G_total += (k_list[i]/K)*Pk_list[i]*G_k[i]
    return G_k, G_total

# ===========================================
# DMFT Simulation
# ===========================================
def DMFT_RNN_torch(params, k_info, hyperparams, save=True, obs=None, ic=None, folder_data='rnn/data3/'):
    g, K, mu, gamma = params
    k_list_raw, Pk_list_raw = k_info
    N, T, dt, N_max, a = hyperparams

    k_list = torch.tensor(k_list_raw, device=device)
    Pk_list = torch.tensor(Pk_list_raw, device=device)
    N_k = len(k_list)

    corr = torch.eye(T, device=device)
    m = torch.zeros(T, device=device)
    G = torch.zeros((T,T), device=device)
    G_k = torch.zeros((N_k,T,T), device=device)
    if gamma!=0: 
        tril_mask = torch.tril(torch.ones((T,T), device=device))*0.1
        for i in range(N_k): 
            G_k[i] = tril_mask
            G += (k_list[i]/K)*Pk_list[i]*G_k[i]

    x0 = torch.empty(N_k, N, device=device).uniform_(-1,1) if ic is None else ic
    x = torch.zeros((N_k,N,T), device=device)
    sqrt_kK = torch.sqrt(k_list/K).unsqueeze(1)
    k_over_K = (k_list/K).unsqueeze(1)
    err_list, obs_list = [], []

    for iter in range(N_max):
        eta = torch.stack([sample_multivariate_gaussian_torch(torch.zeros(T, device=device), corr, N) for _ in range(N_k)])
        x[:,:,0] = x0
        for t in range(1,T):
            x_prev = x[:,:,t-1]
            noise_term = g*sqrt_kK*eta[:,:,t-1]
            drive_term = mu*m[t-1]*k_over_K
            rec_term = 0
            if gamma!=0:
                phi_past = phi(x[:,:, :t])
                G_row = G[t-1,:t].view(1,1,-1)
                rec_term = gamma*g**2*k_over_K*(dt*(G_row*phi_past)).sum(dim=2)
            x[:,:,t] = x_prev + dt*(-x_prev + noise_term + drive_term + rec_term)

        phi_x = phi(x)
        corr_k, corr_new = compute_corr_k(phi_x, k_list, Pk_list, K)
        m_new = compute_magnetization(phi_x, k_list, Pk_list, K)
        if gamma!=0: G_k_new, G_new = compute_G_k(x, G_k, G, k_list, Pk_list, K, dt, g, gamma)
        else: G_k_new, G_new = None, None

        err_corr = (corr_new-corr).norm()
        err_m = (m_new-m).norm()
        err_G = (G_new-G).norm() if gamma!=0 else 0.0
        err_list.append((err_corr.item(), err_m.item(), err_G))
        print(f"Iter {iter+1}/{N_max} | Error: C={err_corr:.2e}, M={err_m:.2e}, G={err_G:.2e}")

        corr = (1-a)*corr + a*corr_new
        m = (1-a)*m + a*m_new
        if gamma!=0:
            G = (1-a)*G + a*G_new
            G_k = (1-a)*G_k + a*G_k_new

        obs_list.append((m[-1].cpu(), corr[-1,-1].cpu(), (dt*G[-1,:].sum()).cpu() if gamma!=0 else None))

        if save:
            torch.save({'n_iter': iter, 'x': x.clone(), 'corr_k': corr_k.clone(),
                        'err_list': err_list, 'm': m.clone(), 'corr': corr.clone(),
                        'obs_list': obs_list.copy(),
                        'G_k': G_k.clone() if G_k is not None else 0,
                        'G': G.clone() if G is not None else 0}, folder_data+f'last_iteration_DMFT_RNN_gamma{gamma}.pt')

        if err_corr < 1e-12: break
        gc.collect(); torch.cuda.empty_cache()

    return x.cpu(), corr_k.cpu(), err_list, eta.cpu(), m.cpu(), corr.cpu(), obs_list, (G_k.cpu() if G_k is not None else None), (G.cpu() if G is not None else None)

# ===========================================
# Plotting Functions
# ===========================================
def plot_neuron_trajectories(x, n_random=5):
    x = x.cpu().numpy() if torch.is_tensor(x) else x
    N = x.shape[1]
    plt.figure(figsize=(7,5))
    for i in np.random.randint(0,N,n_random): plt.plot(x[0,i,:], label=f"Neuron {i+1}")
    plt.xlabel("Time step"); plt.ylabel("Neuron state"); plt.title("Random Neuron Trajectories")
    plt.grid(True); plt.tight_layout(); plt.show()

def plot_magnetization(m):
    m = m.cpu().numpy() if torch.is_tensor(m) else m
    plt.figure(figsize=(7,5)); plt.plot(m, lw=2)
    plt.xlabel("Time step"); plt.ylabel("Magnetization"); plt.title("Magnetization Over Time")
    plt.grid(True); plt.tight_layout(); plt.show()

def plot_convergence_error(err_list):
    err_array = np.array([[v.item() if torch.is_tensor(v) else v for v in e] for e in err_list])
    labels = ["Correlation", "Magnetization", "Response"]
    plt.figure(figsize=(7,5))
    for i in range(err_array.shape[1]): plt.plot(err_array[:,i], label=labels[i])
    plt.xlabel("Iteration"); plt.ylabel("Error"); plt.yscale("log"); plt.title("DMFT Convergence Error")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

def plot_correlation_matrix(corr):
    corr = corr.cpu().numpy() if torch.is_tensor(corr) else corr
    plt.figure(figsize=(7,7)); plt.imshow(np.flipud(corr), cmap='hot', origin='lower')
    plt.colorbar(label="Correlation"); plt.xlabel("t'"); plt.ylabel("t"); plt.title("Correlation Matrix")
    plt.tight_layout(); plt.show()

def plot_response_function(G, dt):
    if G is None: return
    G_np = G.cpu().numpy() if torch.is_tensor(G) else G
    plt.figure(figsize=(7,7)); plt.imshow(np.flipud(G_np.T), cmap='viridis')
    plt.colorbar(label="G(t,t')"); plt.xlabel("t'"); plt.ylabel("t"); plt.title("Response Function G(t,t')")
    plt.tight_layout(); plt.show()

    T = G_np.shape[0]; tau_min = int(0.7*T); lags = np.arange(1,T-tau_min); G_lag = np.zeros_like(lags)
    for i, tau in enumerate(lags): t_vals = np.arange(tau_min, T-tau_min); G_lag[i] = np.mean(G_np[t_vals+tau,t_vals])
    plt.figure(figsize=(6,4)); plt.plot(lags*dt, G_lag, color='firebrick', lw=2); plt.xlabel(r'$\tau$'); plt.ylabel(r'$G(\tau)$'); plt.title("Response Function vs Time Lag"); plt.grid(True); plt.tight_layout(); plt.show()

def plot_final_state_distribution(x):
    x = x.cpu().numpy() if torch.is_tensor(x) else x
    plt.figure(figsize=(7,5)); plt.hist(x[0,:,-1], bins=100, density=True, alpha=0.7, color='skyblue')
    plt.xlabel("Neuron state"); plt.ylabel("Density"); plt.title("Final State Distribution"); plt.grid(True); plt.tight_layout(); plt.show()

def plot_corr_vs_degree(corr_k, k_vals):
    plt.figure(figsize=(7,5))
    for i in range(corr_k.shape[0]):
        data = corr_k[i,-1]; data = data.cpu().numpy()[::-1] if torch.is_tensor(data) else data[::-1]
        tau_lag = HWHM(data)
        plt.plot(data, label=f'$k={k_vals[i]}$')
        if tau_lag is not None: plt.axvline(x=tau_lag, color='red', linestyle='--', alpha=0.5)
    plt.xlabel(r'$\tau$'); plt.ylabel(r'$C(\tau)$'); plt.title("Correlation Function per Degree"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

#### Simulating ODE to use as initial conditions

In [ ]:
# ===========================================
# Imports
# ===========================================
import numpy as np
import torch
from collections import defaultdict

# ===========================================
# Parameters
# ===========================================
k_list = np.array([100, 1000])
Pk_list = np.array([0.9, 0.1])

N = 1_000
N_steps = 2_000
dt = 0.05
K = np.sum(k_list * Pk_list)

g = 3
gamma = 0.1
mu = 0
n = 0

params = [N, g, n, mu, gamma, k_list, Pk_list]
hyperparams = [N_steps, dt]

n_replicas = 100      # Number of degree sequences / ICs to generate
n_samples = 10_000    # Number of ICs per degree

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ===========================================
# Simulate multiple degree sequences and ICs
# ===========================================
degrees_list = []
ics_list = []

for i in range(n_replicas):
    print(f"Simulation {i+1}/{n_replicas}")
    degree_sequence, ic = simulate_costum_degree(params, hyperparams)
    degrees_list.append(degree_sequence)
    ics_list.append(ic)

# Flatten arrays
degrees = np.concatenate(degrees_list)
ics = np.concatenate(ics_list)

# ===========================================
# Group ICs by degree
# ===========================================
degree_to_ics = defaultdict(list)
for deg, ic in zip(degrees, ics):
    degree_to_ics[int(deg)].append(ic)

# Convert lists to arrays
degree_to_ics = {k: np.array(v) for k, v in degree_to_ics.items()}
unique_degrees = sorted(degree_to_ics.keys())

print("Grouped ICs shape:", {k: v.shape for k, v in degree_to_ics.items()})
print("Counts per degree:", {k: len(v) for k, v in degree_to_ics.items()})

# ===========================================
# Sample ICs per degree
# ===========================================
x0_list = []
for deg in unique_degrees:
    ic_array = degree_to_ics[deg].flatten()  # flatten multidimensional ICs
    sampled = np.random.choice(ic_array, size=n_samples, replace=True)
    x0_list.append(sampled)

# Stack into final tensor: shape [num_degrees, n_samples]
x0 = torch.tensor(np.stack(x0_list, axis=0), dtype=torch.float32, device=device)
print("x0 shape:", x0.shape)  # [num_degrees, n_samples]

# ===========================================
# Save and cleanup
# ===========================================
torch.save(x0, "x0_gamma0.1_g3.pt")
del x0
torch.cuda.empty_cache()

# ===========================================
# Load initial conditions
# ===========================================
x0_loaded = torch.load("x0_gamma0.1_g3.pt", map_location=device)
print("Loaded x0 shape:", x0_loaded.shape)

#### DMFT solution

In [ ]:
# ===========================================
# 1. Simulation Parameters
# ===========================================
import torch
import gc

g = 3                 # gain
mu = 0                # average coupling
k_list = np.array([100, 1000])
Pk_list = np.array([0.9, 0.1])
K = np.sum(k_list * Pk_list)
k_info = (k_list, Pk_list)

# DMFT hyperparameters
N = 10_000       # neurons per group
T = 600          # time steps
dt = 0.2         # time step size
N_max = 200      # max DMFT iterations
a = 0.3          # DMFT update rate
hyperparams = (N, T, dt, N_max, a)

# List of correlation parameters
gamma_list = [0, 0.05, 0.1, 0.15]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float32)
print("Using device:", device)

# ===========================================
# 2. Run DMFT for each gamma
# ===========================================
dmft_results = {}  # store results for each gamma

for gamma in gamma_list:
    print(f"\nRunning DMFT for gamma = {gamma}")
    params = (g, K, mu, gamma)

    # Clear GPU/CPU memory
    gc.collect()
    torch.cuda.empty_cache()

    # Run DMFT
    results = DMFT_RNN_torch(params, k_info, hyperparams, ic=None)
    dmft_results[gamma] = results

    x, corr_k, err_list, eta, m, corr, obs_list, G_k, G = results
    print(f"DMFT finished for gamma={gamma}")

# ===========================================
# 3. Load previous result (optional)
# ===========================================
# results = torch.load('/PLACEHOLDER/data/last_iteration_DMFT_RNN_gamma0.1.pt')
# x = results['x']
# corr_k = results['corr_k']
# err_list = results['err_list']
# m = results['m']
# corr = results['corr']
# obs_list = results['obs_list']
# G_k = results['G_k']
# G = results['G']

# ===========================================
# 4. Generate plots for the last gamma
# ===========================================
plot_neuron_trajectories(x, n_random=5)
plot_magnetization(m)
plot_convergence_error(err_list)
plot_correlation_matrix(corr)
plot_response_function(G, dt)
plot_final_state_distribution(x)
plot_corr_vs_degree(corr_k, k_list)

#### Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch

# -------------------------------
# Utility: degree-dependent colors
# -------------------------------
def degree_colors(k_vals, base_color, light_color=None):
    k_vals = np.array(k_vals)
    base = np.array(mpl.colors.to_rgb(base_color))
    if light_color is None:
        light_color = base + (1 - base) * 0.5
    light_color = np.clip(light_color, 0, 1)
    norm = mpl.colors.Normalize(vmin=min(k_vals), vmax=max(k_vals))
    cmap = mpl.colors.LinearSegmentedColormap.from_list("degree_map", [light_color, base])
    return [cmap(norm(k)) for k in k_vals]

# -------------------------------
# Convert observables to array
# -------------------------------
obs_array = np.array([[v.item() if torch.is_tensor(v) else v for v in obs] for obs in obs_list])
if torch.is_tensor(corr):
    corr = corr.cpu().numpy()
if G is not None and torch.is_tensor(G):
    G_np = G.cpu().numpy()
else:
    G_np = G

# -------------------------------
# Figure positions and labels
# -------------------------------
fig = plt.figure(figsize=(18/2.54, 10/2.54))
positions = {
    "a": [0.08, 0.57, 0.22, 0.32],
    "b": [0.3, 0.57, 0.36, 0.32],
    "c": [0.75, 0.57, 0.22, 0.32],
    "d": [0.08, 0.12, 0.22, 0.32],
    "e": [0.3, 0.12, 0.36, 0.32],
    "f": [0.75, 0.12, 0.22, 0.32],
}
subplot_labels = ["a)", "b)", "c)", "d)", "e)", "f)"]

colors = ['#D64444', '#43A4C4']  # for k_list=[100, 1000]

# =========================
# a) Observables: C[t,t]
# =========================
ax_a = fig.add_axes(positions["a"])
ax_a.plot(obs_array[:, 1], color=plt.cm.magma(0.8), lw=2)
ax_a.set_xlabel("Iteration")
ax_a.set_ylabel("C[t,t]")
ax_a.set_xticks([0, 100, 200])
ax_a.set_yticks([0.88, 0.92, 0.96])
ax_a.text(-0.2, 1.1, subplot_labels[0], transform=ax_a.transAxes, fontweight='bold')

# =========================
# b) Correlation matrix
# =========================
ax_b = fig.add_axes(positions["b"])
im_b = ax_b.imshow(corr, cmap="magma", origin="lower", aspect='equal')
ax_b.set_xlabel(r"$t'$")
ax_b.set_ylabel(r"$t$")
ax_b.set_xticks([0, 200, 400])
ax_b.set_yticks([0, 200, 400])
ax_b.invert_yaxis()
ax_b.text(-0.2, 1.1, subplot_labels[1], transform=ax_b.transAxes, fontweight='bold')

cax_b = fig.add_axes([positions["b"][0] + positions["b"][2] - 0.05,
                      positions["b"][1], 0.012, positions["b"][3]])
plt.colorbar(im_b, cax=cax_b, label=r"$C[t, t']$")

# =========================
# c) Correlation per degree
# =========================
ax_c = fig.add_axes(positions["c"])
for i, (k, color) in enumerate(zip(k_list, colors)):
    reversed_data = corr_k[i, -1]
    if torch.is_tensor(reversed_data):
        reversed_data = reversed_data.cpu().numpy()
    reversed_data = reversed_data[::-1]
    tau_lag = HWHM(reversed_data)

    ax_c.plot(reversed_data, color=color, lw=2, label=fr"$k={k}$")
    if tau_lag is not None:
        ax_c.axvline(x=tau_lag, color=color, linestyle="--", alpha=0.4)

ax_c.set_xlabel(r"$\tau$")
ax_c.set_xlim(0, 400)
ax_c.set_xticks([0, 200, 400])
ax_c.set_ylabel(r"$C_k(\tau)$")
ax_c.set_yticks([0, 0.5, 1])
ax_c.legend(loc="upper right", frameon=False)
ax_c.text(-0.2, 1.1, subplot_labels[2], transform=ax_c.transAxes, fontweight='bold')

# =========================
# d) Observable: response magnitude
# =========================
ax_d = fig.add_axes(positions["d"])
ax_d.plot(obs_array[:, 2], color=plt.cm.cividis(0.8), lw=2)
ax_d.set_xlabel("Iteration")
ax_d.set_ylabel(r"$\chi$")
ax_d.set_xticks([0, 100, 200])
ax_d.set_yticks([2, 5, 8])
ax_d.text(-0.15, 1.05, subplot_labels[3], transform=ax_d.transAxes, fontweight='bold')

# =========================
# e) Response matrix
# =========================
ax_e = fig.add_axes(positions["e"])
im_e = ax_e.imshow(G_np, cmap="cividis", origin="lower", aspect='equal')
ax_e.set_xlabel(r"$t'$")
ax_e.set_ylabel(r"$t$")
ax_e.set_xticks([0, 200, 400])
ax_e.set_yticks([0, 200, 400])
ax_e.invert_yaxis()
ax_e.text(-0.2, 1.1, subplot_labels[4], transform=ax_e.transAxes, fontweight='bold')

cax_e = fig.add_axes([positions["e"][0] + positions["e"][2] - 0.05,
                      positions["e"][1], 0.012, positions["e"][3]])
plt.colorbar(im_e, cax=cax_e, label=r"$G[t, t']$")

# =========================
# f) Response vs lag per degree
# =========================
ax_f = fig.add_axes(positions["f"])
tau_min_frac = 0.7
T = G_np.shape[0]
tau_min = int(tau_min_frac * T)
lags = np.arange(1, T - tau_min)

G_np_k = np.array([g.cpu().numpy() if torch.is_tensor(g) else g for g in G_k])

for i, (k, color) in enumerate(zip(k_list, colors)):
    G_lag_k = np.zeros_like(lags, dtype=np.float32)
    for j, tau in enumerate(lags):
        t_vals = np.arange(tau_min, T - tau)
        G_lag_k[j] = np.mean(G_np_k[i, t_vals + tau, t_vals])
    ax_f.plot(lags * dt, G_lag_k, color=color, lw=2, label=fr"$k={k}$")

ax_f.set_xlabel(r"$\tau$")
ax_f.set_ylabel(r"$G_k(\tau)$")
ax_f.set_xlim(0, 10)
ax_f.set_xticks([0, 5, 10])
ax_f.legend(loc="upper right", frameon=False)
ax_f.text(-0.2, 1.1, subplot_labels[5], transform=ax_f.transAxes, fontweight='bold')

plt.savefig('PLACEHOLDER/dmft_gamma0.1.png', dpi=500, bbox_inches='tight')
plt.show()

# -------------------------------
# Trajectories and stationary distributions
# -------------------------------
labels = ['k=100', 'k=1000']
fig, axes = plt.subplots(1, 2, figsize=(18/2.54, 6/2.54))

# Left: example trajectories (first 5 neurons per degree)
for i in range(2):
    for traj in x[i, :5, :].cpu().numpy():
        axes[0].plot(traj, alpha=0.7, color=colors[i])
axes[0].set_xlabel("Time")
axes[0].set_xticks([0, 200, 400])
axes[0].set_ylabel("Activity")
axes[0].text(-0.1, 1.05, "a)", transform=axes[0].transAxes, fontweight='bold')

# Right: stationary distributions
for i in range(2):
    hist_data = x[i, :, -1].cpu().numpy()
    axes[1].hist(hist_data, bins=50, density=True, alpha=0.7, color=colors[i], label=labels[i])
axes[1].set_xlabel("Activity")
axes[1].set_ylabel("Probability density")
axes[1].text(-0.1, 1.05, "b)", transform=axes[1].transAxes, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('PLACEHOLDER/dmft_traj_gamma0.1.png', dpi=500, bbox_inches='tight')
plt.show()

#### Approximated self-coupling strength

In [ ]:
chi = obs_array[-1,2]
print('Susceptibility chi:', chi)

g = 3
gamma = 0.1

s1 = g**2*k_list[0]/K*gamma*chi
s2 = g**2*k_list[1]/K*gamma*chi

print('s1:', s1)
print('s2:', s2)

# Libraries
import numpy as np
import torch
import matplotlib.pyplot as plt

# Simple Euler integrator for RNN in PyTorch
def integrate_RNN(J, N_steps, dt, x0=None):
    N = J.shape[0]
    if x0 is None:
        x = torch.randn(N, device=J.device)
    else:
        x = x0.clone()
    
    trajectory = torch.zeros((N_steps, N), device=J.device)
    for t in range(N_steps):
        dxdt = -x + J @ torch.tanh(x)
        x = x + dxdt * dt
        trajectory[t] = x
    return trajectory.cpu().numpy()  # Convert to numpy for ACF calculation

# Parameters
N = 2000       # Number of neurons
g = 3          # Gain
N_steps = 20_000  # Number of time steps
dt = 0.2     # Time step size
# Self-couplings
f1 = Pk_list[0]
f2 = Pk_list[1]
# Compute ACF values
N_stat = 2_000
N_lags = 12_000
# Number of trajectories
N_samples = 10

# Using statsmodels for ACF
from statsmodels.tsa.stattools import acf

# Storing ACFs
acf1 = []
acf2 = []

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on {device}')

for n in range(N_samples):
    print(f'Simulating: {n+1}/{N_samples}')
    J = g * torch.randn(N, N, device=device) / torch.sqrt(torch.tensor(N, dtype=torch.float32))
    # Add self-couplings
    diag_vals = torch.cat([torch.ones(int(f1*N), device=device) * s1,
                           torch.ones(int(f2*N), device=device) * s2])
    J = J - torch.diag(torch.diag(J)) + torch.diag(diag_vals)
    
    trajectory = integrate_RNN(J, N_steps, dt)
    
    for i in range(int(f1*N)):
        acf1.append(acf(trajectory[N_stat:, i], nlags=N_lags))
    for i in range(int(f1*N), N):
        acf2.append(acf(trajectory[N_stat:, i], nlags=N_lags))

acf1_mean = np.mean(acf1, axis=0)
acf2_mean = np.mean(acf2, axis=0)

HWHM1 = np.where(acf1_mean < 0.5)[0][0] 
HWHM2 = np.where(acf2_mean < 0.5)[0][0] 


plt.figure(figsize=(14/2.54, 8/2.54))

# Muted colors for DMFT curves
colors_dmft = ['#D64444', '#43A4C4']

# Plot DMFT-derived correlations as dashed lines
for i, color in zip(range(corr_k.shape[0]), colors_dmft):
    data = corr_k[i, -1]
    if torch.is_tensor(data):
        data = data.cpu().numpy()
    data = data[::-1]  # Reverse if needed

    plt.plot(data/data[0], label=fr"DMFT $k={k_list[i]}$", color=color, linestyle='--', lw=2)

# Plot approximate self-coupling results as solid lines
# Match S1 with k=100, S2 with k=1000
idx_s1 = np.argmin(np.abs(np.array(k_list) - 100))
idx_s2 = np.argmin(np.abs(np.array(k_list) - 1000))

plt.plot(np.arange(N_lags+1), acf1_mean, label=r'Approx $S_1\simeq 0.5$', color=colors_dmft[idx_s1], lw=2)
plt.plot(np.arange(N_lags+1), acf2_mean, label=r'Approx $S_2\simeq 5$', color=colors_dmft[idx_s2], lw=2)

# Axis limits
plt.xlim(0, 300)

# Reduce number of ticks
plt.xticks(np.arange(0, 301, 50))
plt.yticks(np.linspace(0, 1, 5))

# Labels
plt.xlabel(r"Lag $\tau$ [timesteps]" , fontsize=12)
plt.ylabel(r"$C(\tau)$", fontsize=12)

# Legend
plt.legend(frameon=False, fontsize=11)

plt.tight_layout()
plt.savefig(f'PLACEHOLDER/case_study_comparison_DMFT_approx.png', dpi=500, bbox_inches='tight')


#### Comparison between timescales

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# -------------------------------
# Degree and file setup
# -------------------------------
k_list = np.array([100, 1000])
Pk_list = np.array([0.9, 0.1])

files = [
    'last_iteration_DMFT_RNN_gamma0.0.pt',
    'last_iteration_DMFT_RNN_gamma0.05.pt',
    'last_iteration_DMFT_RNN_gamma0.1.pt',
    'last_iteration_DMFT_RNN_gamma0.15.pt'
]
gamma_list = [0., 0.05, 0.1, 0.15]

corr_k_list = []
G_k_list = []

# -------------------------------
# HWHM function
# -------------------------------
def HWHM(y):
    """Estimate the half-width at half-maximum (HWHM) of a 1D curve y."""
    half_max = 0.5 * (np.max(y) - np.min(y)) + np.min(y)
    for i, val in enumerate(y):
        if val <= half_max:
            return i
    return len(y)//2

# -------------------------------
# Load DMFT results and process
# -------------------------------
for file in files:
    print('Processing file:', file)
    results = torch.load('PLACEHOLDER/' + file)
    corr_k = results['corr_k']
    G_k = results['G_k']
    obs_list = results['obs_list']
    dt = 0.2  # time step

    # Convert obs_list to numpy safely
    obs_array = np.array([[v.item() if torch.is_tensor(v) else v for v in obs] for obs in obs_list])

    # -------------------------------
    # Correlation function per degree
    # -------------------------------
    for i in range(corr_k.shape[0]):
        data = corr_k[i, -1]
        if torch.is_tensor(data):
            data = data.cpu().numpy()
        data = data[::-1]  # reverse for plotting
        corr_k_list.append(data)

        # Plot HWHM line (optional)
        tau_lag = HWHM(data)
        if tau_lag is not None:
            plt.axvline(x=tau_lag, color='grey', linestyle='--', alpha=0.3)

    # -------------------------------
    # Response function per degree
    # -------------------------------
    if torch.is_tensor(G_k[0]):
        G_np_k = np.array([g.cpu().numpy() for g in G_k])
    else:
        G_np_k = np.array(G_k)

    T = G_np_k.shape[1]
    tau_min_frac = 0.7
    tau_min = int(tau_min_frac * T)
    max_lag = T - tau_min
    lags = np.arange(1, max_lag)

    for i in range(len(k_list)):
        G_lag_k = np.zeros_like(lags, dtype=np.float32)
        for j, tau in enumerate(lags):
            t_vals = np.arange(tau_min, T - tau)
            G_lag_k[j] = np.mean(G_np_k[i, t_vals + tau, t_vals])
        G_k_list.append(G_lag_k)

# -------------------------------
# Plot combined correlations and responses
# -------------------------------
fig, ax = plt.subplots(figsize=(8/2.54, 6/2.54))
fig.subplots_adjust(left=0.13, right=0.98, bottom=0.15, top=0.95)

colors = ['royalblue', 'firebrick', 'forestgreen', 'darkorange']

# Correlation curves (C_k)
for i, gamma in enumerate(gamma_list):
    ax.plot(corr_k_list[2*i] / corr_k_list[2*i][0], linestyle=':', color=colors[i], lw=2)
    ax.plot(corr_k_list[2*i + 1] / corr_k_list[2*i + 1][0], linestyle='--', color=colors[i], lw=2)

# Weighted network response G
K = np.sum(k_list * Pk_list)
g_100 = G_k_list[2] / G_k_list[2][1]
g_1000 = G_k_list[3] / G_k_list[3][1]
G = g_100 * k_list[0]/K * Pk_list[0] + g_1000 * k_list[1]/K * Pk_list[1]
ax.plot(G, linestyle='-', color=colors[2], lw=2.3)

# HWHM markers
tau = np.arange(400)
hwhm_C1000 = [tau[HWHM(corr_k_list[2*i + 1] / corr_k_list[2*i + 1][0])] for i in range(len(gamma_list))]
hwhm_G = tau[HWHM(G)]
x_ticks = hwhm_C1000 + [hwhm_G]

# Axes formatting
ax.set_xscale('log')
ax.set_xticks(x_ticks)
ax.set_xticklabels([f'{(x*0.2):.1f}' for x in x_ticks])
ax.set_xlabel(r"$\tau$")
ax.set_ylabel("Normalized observable")
ax.set_yticks([0, 0.5, 1])
ax.set_ylim(0, 1)
ax.margins(x=0.02)

# Grey vertical HWHM lines
for xt in x_ticks:
    ax.axvline(xt, color='grey', linestyle='-.', lw=1.5, alpha=0.3)

# -------------------------------
# Top-right gamma legend (color boxes)
# -------------------------------
ax_pos = ax.get_position()
legend_y = ax_pos.y1 + 0.02
rect_width, rect_height, spacing = 0.03, 0.03, 0.2
start_x = ax_pos.x0 + 0.0

for i, g in enumerate(gamma_list):
    x_pos = start_x + i*(rect_width + spacing)
    rect = Rectangle((x_pos, legend_y), rect_width, rect_height,
                     transform=fig.transFigure, facecolor=colors[i], edgecolor='none')
    fig.add_artist(rect)
    fig.text(x_pos + rect_width + 0.01, legend_y + rect_height/2,
             fr"$\gamma={g}$", va='center', ha='left', fontsize=11)

# -------------------------------
# Bottom-left line style legend
# -------------------------------
line_labels = [(':', r'$C_{k=100}$'), ('--', r'$C_{k=1000}$'), ('-', r'$G_\mathrm{net}$')]
for i, (style, label) in enumerate(line_labels):
    y_pos = 0.10 + i*0.08
    ax.plot([0.03, 0.11], [y_pos]*2, linestyle=style, color='black', lw=2, transform=ax.transAxes)
    ax.text(0.12, y_pos, label, transform=ax.transAxes, va='center', fontsize=11)

ax.set_xlim(1, 390)

# -------------------------------
# Save figure
# -------------------------------
plt.savefig('PLACEHOLDER/case_study_combined_correlations.png', dpi=600, bbox_inches='tight')
plt.show()